# Option B -- PC-Well Feature-Space Recentering: Sweep

For every (group, model, curve_type, outlier_filter, curve_alignment[, pc_ttp_anchor],
train_center_frac) combination that has actually been trained, treats each of that
group's chips as if it were a brand-new, unseen chip in turn (using the LOFO fold
model that genuinely excluded that chip from training -- not the `--train_full`
model, which would have already seen it), predicts with and without `--pc_recenter`,
and compares against that chip's real labels.

Reuses `08_cross_dataset_predict_new_chip.py`'s own functions (`align_new_chip`,
`predict_new_chip`, `reference_pc_embedding`) via import -- nothing here is a
reimplementation, same pattern as the earlier calibration notebook. `predict_new_chip`
handles both plain curve models and spatial (`cosine_recon`/`attn_recon`) models
transparently -- for spatial models it builds a real neighbor stack from the chip's
own `coords`/`well_ids` for the main curves, and uses a mean-PC-curve-repeated stack
for `pc_recenter`'s embeddings (no real PC spatial coordinates needed -- see `08`'s
`_pc_mean_stack` docstring for why that's exact, not approximate).

**`train_center_frac` doesn't change which `.keras` file gets loaded** -- per
`04_output_map.md` §3b, only the *training pool* is center-cropped, and the saved
model still lands at the same unscoped path. It only changes which results-joblib
subfolder `class_names`/metrics get read from (some filters, e.g. `noamp_remove` in
these groups, only ever ran under one `train_center_frac` and have no unscoped
counterpart -- see the Audit section). So two rows differing only in
`train_center_frac` can show identical `acc_baseline`/`acc_pc_recenter` if both
happen to resolve to the same `.keras` file; that's expected, not a bug.

**Combinations that aren't trained yet are skipped, not errored** -- point
`GROUPS_TO_TRY`/`MODELS_TO_TRY`/`OUTLIER_FILTERS_TO_TRY`/`ALIGNMENTS_TO_TRY`/
`TRAIN_CENTER_FRACS_TO_TRY` at whatever you want to compare; missing `.keras` files
or result files just print `[SKIP]` and the sweep continues. Re-run this notebook any
time (e.g. once a job finishes) to pick up newly-trained combinations -- no need to
prune the config lists down to only what exists yet. The Audit section (§3) below
cross-checks these lists against what's actually on disk, so nothing trained gets
silently missed.

In [11]:
import os
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

try:
    notebook_path = globals().get('__vsc_ipynb_file__')
    if notebook_path:
        notebook_dir = os.path.dirname(os.path.dirname(notebook_path))
        os.chdir(notebook_dir)
except Exception as e:
    print(f"Could not change directory: {e}")

print("Current Working Directory:", os.getcwd())
%load_ext autoreload
%autoreload 2
import config

# Import-only -- none of these .py files are modified.
cdt = importlib.import_module("04_cross_dataset_training")
p08 = importlib.import_module("08_cross_dataset_predict_new_chip")
vis07 = importlib.import_module("07_attribution_vis_all")
rio = importlib.import_module("cross_dataset_result_io")

import tensorflow as tf
# DANN/CORAL models were compiled without jit_compile=False (unlike every
# other model type here) -- XLA's cudnn-conv autotuner can fail with a
# spurious RESOURCE_EXHAUSTED even on an idle GPU. Disable globally so
# predicting on already-saved models doesn't hit it.
tf.config.optimizer.set_jit(False)

import joblib

Current Working Directory: /vol/bitbucket/gk225/POC_DDM/gk_code/main
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Configuration -- edit these lists to change what gets compared

In [12]:
GROUPS_TO_TRY = list(config.CROSS_DATASET_GROUPS.keys())[3:7]
EXP_FOLDER = config.FINAL_EXP_FOLDER + "_nc_subtract"
MODE_STR = "lofo"

CURVE_TYPES_TO_TRY = ["ori_curve_norm", "ori_curve_wavelet_bior35_norm"]

MODELS_TO_TRY = [
    "cnn_gru_dual",                          "cnn_gru_dual_dann",                     "cnn_gru_dual_coral",
    "cnn_gru_dual_supcon3",                  "cnn_gru_dual_supcon3_dann",             "cnn_gru_dual_supcon3_coral",
    "cnn_gru_dual_attn_recon",               "cnn_gru_dual_attn_recon_dann",          "cnn_gru_dual_attn_recon_coral",
    "cnn_gru_dual_attn_recon_supcon3",       "cnn_gru_dual_attn_recon_supcon3_dann",  "cnn_gru_dual_attn_recon_supcon3_coral",
]

OUTLIER_FILTERS_TO_TRY = ["none", "lofo_ae", "noamp_remove"]

# (curve_alignment, pc_ttp_anchor) -- anchor is ignored when alignment is acquisition_start
ALIGNMENTS_TO_TRY = [
    ("acquisition_start", "min"),
    ("pc_ttp", "min"),
    ("pc_ttp", "percentile"),
]

# None = full training pool (no center-crop); 0.5 = --train_center_frac 0.5
TRAIN_CENTER_FRACS_TO_TRY = [None, 0.5]

folder_names_by_group = {g: config.CROSS_DATASET_GROUPS[g] for g in GROUPS_TO_TRY}
exp_paths_by_group = {g: [Path(EXP_FOLDER, name) for name in names] for g, names in folder_names_by_group.items()}

def short_name(folder):
    return folder.split('_U_', 1)[1]

n_pairs = sum(len(v) for v in folder_names_by_group.values())
print(f"Groups: {GROUPS_TO_TRY}")
print(f"{n_pairs} (group, chip) pairs x {len(MODELS_TO_TRY)} models x {len(OUTLIER_FILTERS_TO_TRY)} filters "
     f"x {len(ALIGNMENTS_TO_TRY)} alignments x {len(CURVE_TYPES_TO_TRY)} curve_types x "
     f"{len(TRAIN_CENTER_FRACS_TO_TRY)} train_center_fracs = up to "
     f"{n_pairs*len(MODELS_TO_TRY)*len(OUTLIER_FILTERS_TO_TRY)*len(ALIGNMENTS_TO_TRY)*len(CURVE_TYPES_TO_TRY)*len(TRAIN_CENTER_FRACS_TO_TRY)} rows "
     f"(most will be [SKIP]ped until trained).")

Groups: ['final_4_chip_clean_nn', 'final_4_chip_cleanv2_nn', 'final_4_chip_cov_hadv_iav', 'final_4_chip_clean_nn_hpc']
16 (group, chip) pairs x 12 models x 3 filters x 3 alignments x 2 curve_types x 2 train_center_fracs = up to 6912 rows (most will be [SKIP]ped until trained).


In [13]:
EXP_FOLDER

'/vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final_nc_subtract'

## 2. Helpers

`out_dir_for` mirrors `04`/`08`'s own alignment-namespacing exactly (reused logic,
not reimplemented -- just the path-join, since `08`'s functions all take `out_dir`
as a parameter rather than deriving it themselves). `_ALIGN_CACHE` avoids re-aligning
the same chip's curves for every model/filter/train_center_frac that shares the same
(group, curve_type, curve_alignment, anchor) -- alignment doesn't depend on any of
those. It *does* depend on `group_name`: `config.LOFO_EXCLUDE_WELL_MAPPING` excludes
different wells per group, so the same physical chip aligns differently depending on
which group it's being evaluated under.

In [14]:
def out_dir_for(group_name, curve_type, curve_alignment, pc_ttp_anchor):
    out_dir = Path(EXP_FOLDER) / "cross_dataset_cv" / group_name
    if curve_alignment == "pc_ttp":
        out_dir = out_dir / "curve_alignment_pc_ttp" / f"anchor_{pc_ttp_anchor}"
    return out_dir


_ALIGN_CACHE = {}

def aligned_chip(chip_path, group_name, curve_type, curve_alignment, pc_ttp_anchor):
    key = (chip_path.name, group_name, curve_type, curve_alignment, pc_ttp_anchor)
    if key not in _ALIGN_CACHE:
        out_dir = out_dir_for(group_name, curve_type, curve_alignment, pc_ttp_anchor)
        _ALIGN_CACHE[key] = p08.align_new_chip(chip_path, out_dir, curve_type, curve_alignment, pc_ttp_anchor, group_name=group_name)
    return _ALIGN_CACHE[key]


def ground_truth(chip_name, Y_well_raw, class_names):
    mapping = config.LABEL_MAPPINGS[chip_name]
    y_true = np.array([mapping.get(w, w) for w in Y_well_raw])
    # Ground-truth label strings don't always match the model's own class_names
    # (e.g. 'NC-ALL' in LABEL_MAPPINGS vs 'NC' in class_names) -- map by best-effort
    # prefix match against whatever the model actually predicts.
    if class_names:
        cn = list(class_names)
        y_true = np.array([next((c for c in cn if y == c or y.startswith(c + '-') or c.startswith(y + '-')), y)
                           for y in y_true])
    return y_true


print("Helpers defined.")

Helpers defined.


## 3. Audit -- does the config above cover everything trained on disk?

Scans each group's `out_dir` tree directly (independent of the `_TO_TRY` lists above)
for filter-token subfolders that actually hold a results joblib, `center*` subfolders
inside them, and the model-key prefixes baked into `.keras` filenames -- then reports
anything found on disk that isn't already covered by `OUTLIER_FILTERS_TO_TRY`/
`TRAIN_CENTER_FRACS_TO_TRY`/`MODELS_TO_TRY`/`ALIGNMENTS_TO_TRY`. A clean run here
means the sweep in §4 genuinely can't miss a trained combination because of a stale
config list -- not just "I hope I remembered every filter name".

In [15]:
def _has_results(d):
    return any(d.glob("cross_dataset_classification_performances_*.joblib")) or \
           any(d.glob("center*/cross_dataset_classification_performances_*.joblib"))

_known_filter_tokens = {rio.filter_token(f) for f in OUTLIER_FILTERS_TO_TRY}
_known_frac_tokens = {rio._frac_token(f) for f in TRAIN_CENTER_FRACS_TO_TRY if f is not None}
_known_aligns = {a for a, _ in ALIGNMENTS_TO_TRY}
_known_pc_ttp_anchors = {anchor for a, anchor in ALIGNMENTS_TO_TRY if a == "pc_ttp"}
_models_sorted = sorted(MODELS_TO_TRY, key=len, reverse=True)

_unknown = []  # (group, kind, token, which _TO_TRY list to extend)

for group_name in GROUPS_TO_TRY:
    gdir = Path(EXP_FOLDER) / "cross_dataset_cv" / group_name
    if not gdir.exists():
        continue

    found_anchors = {a.name.replace("anchor_", "") for a in gdir.glob("curve_alignment_pc_ttp/anchor_*") if a.is_dir()}
    align_roots = [("acquisition_start", "min", gdir)] + \
                  [("pc_ttp", a, gdir / "curve_alignment_pc_ttp" / f"anchor_{a}") for a in found_anchors]

    for align, anchor, root in align_roots:
        filt_dirs = [d for d in root.iterdir() if d.is_dir()] if root.exists() else []
        filt_dirs = [d for d in filt_dirs if _has_results(d)]
        if not filt_dirs:
            continue  # nothing actually trained under this alignment root

        if align == "acquisition_start" and "acquisition_start" not in _known_aligns:
            _unknown.append((group_name, "curve_alignment", "acquisition_start", "ALIGNMENTS_TO_TRY"))
        if align == "pc_ttp" and anchor not in _known_pc_ttp_anchors:
            _unknown.append((group_name, "pc_ttp anchor", anchor, "ALIGNMENTS_TO_TRY"))

        for filt_dir in filt_dirs:
            if filt_dir.name not in _known_filter_tokens:
                _unknown.append((group_name, "filter token", filt_dir.name, "OUTLIER_FILTERS_TO_TRY"))
            for frac_dir in filt_dir.glob("center*"):
                if frac_dir.is_dir() and frac_dir.name not in _known_frac_tokens:
                    _unknown.append((group_name, "frac token", frac_dir.name, "TRAIN_CENTER_FRACS_TO_TRY"))

        for keras_path in root.glob("model_interpretation/*/*.keras"):
            stem = keras_path.name[:-len("_model.keras")]
            if not any(stem.startswith(m + "_") for m in _models_sorted):
                _unknown.append((group_name, "unrecognized .keras prefix", keras_path.name, "MODELS_TO_TRY"))

if _unknown:
    print("[!] Found on disk but NOT covered by the config lists in section 1:")
    for group_name, kind, token, list_name in sorted(set(_unknown)):
        print(f"    group={group_name:28s} {kind:26s} '{token}'  -> add to {list_name}")
else:
    print("[OK] Every filter/frac/alignment/model token found on disk for these groups is already covered.")

[OK] Every filter/frac/alignment/model token found on disk for these groups is already covered.


## 4. Sweep

For each group, curve_type, alignment, train_center_frac: load whatever results
exist (§3 already confirmed nothing trained is missing from the lists below). For
each filter, model, chip: find the LOFO fold model that excluded that chip
(`model_interpretation/lofo_{chip}/`) -- a genuine held-out test, not resubstitution
against a `--train_full` model that already saw the chip. Skips (silently, given how
large the combination space now is) whenever: the model file doesn't exist yet, or
the chip has no PC snapshot (`--pc_recenter` needs one). `cosine_recon`/`attn_recon`
models are supported -- `predict_new_chip` auto-detects them from the model key and
builds the spatial neighbor-stack itself (same function `08`'s own CLI uses), using
the chip's own `coords`/`well_ids` for the main curves and the mean-PC-curve trick
for `pc_recenter`'s reference/new-chip embeddings (no real PC spatial coords needed).

Well-exclusion policy differs per group (`config.LOFO_EXCLUDE_WELL_MAPPING`), so the
same physical chip is aligned separately for each group it appears in -- `aligned_chip`'s
cache key includes `group_name` for exactly this reason.

In [16]:
GROUPS_TO_TRY

['final_4_chip_clean_nn',
 'final_4_chip_cleanv2_nn',
 'final_4_chip_cov_hadv_iav',
 'final_4_chip_clean_nn_hpc']

In [17]:
rows = []

for group_name in GROUPS_TO_TRY:
    exp_paths_all = exp_paths_by_group[group_name]

    for curve_type in CURVE_TYPES_TO_TRY:
        for curve_alignment, pc_ttp_anchor in ALIGNMENTS_TO_TRY:
            out_dir = out_dir_for(group_name, curve_type, curve_alignment, pc_ttp_anchor)

            for train_center_frac in TRAIN_CENTER_FRACS_TO_TRY:
                lofo_results = p08.load_partitioned(out_dir, MODE_STR, curve_type, train_center_frac=train_center_frac)
                if not any(k != "full_data" for k in lofo_results):
                    continue  # full_data alone (always merged in regardless of frac) isn't real per-fold LOFO results

                for filter_key_raw in OUTLIER_FILTERS_TO_TRY:
                    filter_key = "None" if filter_key_raw.lower() == "none" else filter_key_raw

                    for model_key in MODELS_TO_TRY:
                        for chip_path in exp_paths_all:
                            chip_name = chip_path.name
                            fold_label = f"lofo_{chip_name}"
                            model_dir = out_dir / "model_interpretation" / fold_label
                            model_path = model_dir / f"{model_key}_{filter_key}_{curve_type}_model.keras"
                            if not model_path.exists():
                                continue

                            class_names = lofo_results.get(fold_label, {}).get("class_names")
                            if class_names is None:
                                continue  # .keras existing isn't enough -- no results entry for this exact fold/frac means no real class_names to compare against

                            align_result = aligned_chip(chip_path, group_name, curve_type, curve_alignment, pc_ttp_anchor)
                            if align_result is None:
                                continue
                            curves, resampler, Y_well_raw, pc_curves_aligned, coords, well_ids = align_result

                            loaded = vis07.load_saved_models(
                                model_dir, filter_key, len(resampler.t_grid), curve_type=curve_type, model_names=[model_key])
                            if model_key not in loaded:
                                continue
                            model = loaded[model_key]

                            if p08._is_spatial(model_key) and (coords is None or well_ids is None):
                                continue

                            y_true = ground_truth(chip_name, Y_well_raw, class_names)
                            valid = y_true != "PC"

                            exp_paths_train = [p for p in exp_paths_all if p.name != chip_name]

                            probs_base, _ = p08.predict_new_chip(
                                model, model_key, curves, coords, well_ids, pc_curves_aligned,
                                exp_paths_train, out_dir, curve_type, filter_key, curve_alignment,
                                pc_recenter=False)
                            pred_base = np.array(class_names)[np.argmax(probs_base, axis=1)]
                            acc_base = (pred_base[valid] == y_true[valid]).mean() if valid.any() else float('nan')

                            acc_recenter = float('nan')
                            shift_norm = float('nan')
                            try:
                                probs_r, shift_norm = p08.predict_new_chip(
                                    model, model_key, curves, coords, well_ids, pc_curves_aligned,
                                    exp_paths_train, out_dir, curve_type, filter_key, curve_alignment,
                                    pc_recenter=True, force_rerun=True)
                                pred_r = np.array(class_names)[np.argmax(probs_r, axis=1)]
                                acc_recenter = (pred_r[valid] == y_true[valid]).mean() if valid.any() else float('nan')
                            except ValueError:
                                pass  # no PC snapshot for this chip -- leave acc_recenter as NaN

                            rows.append({
                                "group": group_name,
                                "curve_type": curve_type, "curve_alignment": curve_alignment,
                                "pc_ttp_anchor": pc_ttp_anchor if curve_alignment == "pc_ttp" else "-",
                                "train_center_frac": train_center_frac if train_center_frac is not None else "-",
                                "outlier_filter": filter_key_raw, "model": model_key,
                                "held_out_chip": short_name(chip_name),
                                "n_pixels": int(valid.sum()), "acc_baseline": acc_base,
                                "acc_pc_recenter": acc_recenter, "recenter_delta": acc_recenter - acc_base,
                                "shift_norm": shift_norm,
                            })
                            print(f"  [OK] group={group_name:26s} {model_key:38s} filter={filter_key_raw:14s} "
                                 f"frac={str(train_center_frac):5s} "
                                 f"align={curve_alignment}/{pc_ttp_anchor if curve_alignment=='pc_ttp' else '-':10s} "
                                 f"chip={short_name(chip_name):10s} base={acc_base*100:5.1f}% recenter={acc_recenter*100:5.1f}%")

                            # Every iteration loads a fresh .keras file -- without this, TF's graph
                            # state accumulates across the whole sweep (hundreds of models over 4
                            # groups x 2 fracs x 3 filters x 12 models) until even a tiny cudnn
                            # workspace allocation starts failing on an otherwise-idle GPU.
                            del model, loaded
                            tf.keras.backend.clear_session()

results_df = pd.DataFrame(rows)
print(f"\n{len(results_df)} trained combinations found and evaluated.")

  [*] LOFO_EXCLUDE_WELL_MAPPING['final_4_chip_clean_nn']: dropping 1135 samples from wells [8, 9] (y_label={8: 'PC', 9: 'NC-ALL'}, y_concentration={8: '0e+00', 9: '0e+00'}) for D20260806_E00_C00_F4500KHz_U_DDM_01_06
  [PC-TTP align] new chip TTP=332.81  anchor=79.67  shift=253.15
  [*] Saved PC reference embedding -> /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final_nc_subtract/cross_dataset_cv/final_4_chip_clean_nn/curve_alignment_pc_ttp/anchor_min/pc_reference_embedding_cnn_gru_dual_attn_recon_noamp_remove_ori_curve_norm.joblib
  [OK] group=final_4_chip_clean_nn      cnn_gru_dual_attn_recon                filter=noamp_remove   frac=0.5   align=pc_ttp/min        chip=DDM_01_06  base= 27.9% recenter= 43.8%
  [*] Saved PC reference embedding -> /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final_nc_subtract/cross_dataset_cv/final_4_chip_clean_nn/curve_alignment_pc_ttp/anchor_min/pc_reference_embedding_cnn_gru_dual_attn_recon_dann_noamp_remove_ori_curve_norm.joblib
  [OK] group=final_4

## 5. Results

Per-(chip, combination) rows first, then aggregated by combination (mean across each
group's chips) -- sorted so the biggest recentering wins float to the top. A
combination only appears here once it's actually been trained; re-run the audit and
sweep above after a job finishes to pick up more.

In [18]:
results_df.sort_values(["group", "model", "outlier_filter", "curve_alignment", "held_out_chip"])

,group,curve_type,curve_alignment,pc_ttp_anchor,train_center_frac,outlier_filter,model,held_out_chip,n_pixels,acc_baseline,acc_pc_recenter,recenter_delta,shift_norm
0,final_4_chip_clean_nn,ori_curve_norm,pc_ttp,min,0.5,noamp_remove,cnn_gru_dual_attn_recon,DDM_01_06,13372,0.278717,0.437556,0.158839,15.491244
6,final_4_chip_clean_nn,ori_curve_wavelet_bior35_norm,pc_ttp,min,0.5,noamp_remove,cnn_gru_dual_attn_recon,DDM_01_06,13372,0.159961,0.264732,0.104771,17.167343
2,final_4_chip_clean_nn,ori_curve_norm,pc_ttp,min,0.5,noamp_remove,cnn_gru_dual_attn_recon_coral,DDM_01_06,13372,0.201167,0.139097,-0.062070,4.062100
8,final_4_chip_clean_nn,ori_curve_wavelet_bior35_norm,pc_ttp,min,0.5,noamp_remove,cnn_gru_dual_attn_recon_coral,DDM_01_06,13372,0.433144,0.310948,-0.122196,5.132866
1,final_4_chip_clean_nn,ori_curve_norm,pc_ttp,min,0.5,noamp_remove,cnn_gru_dual_attn_recon_dann,DDM_01_06,13372,0.303395,0.510096,0.206701,49.754662
...,...,...,...,...,...,...,...,...,...,...,...,...,...
33,final_4_chip_cov_hadv_iav,ori_curve_wavelet_bior35_norm,pc_ttp,min,0.5,noamp_remove,cnn_gru_dual_attn_recon_supcon3,DDM_01_06,9544,0.430742,0.605302,0.174560,22.843815
29,final_4_chip_cov_hadv_iav,ori_curve_norm,pc_ttp,min,0.5,noamp_remove,cnn_gru_dual_attn_recon_supcon3_coral,DDM_01_06,9544,0.575335,0.690591,0.115256,8.149722
35,final_4_chip_cov_hadv_iav,ori_curve_wavelet_bior35_norm,pc_ttp,min,0.5,noamp_remove,cnn_gru_dual_attn_recon_supcon3_coral,DDM_01_06,9544,0.518965,0.605302,0.086337,7.783801
28,final_4_chip_cov_hadv_iav,ori_curve_norm,pc_ttp,min,0.5,noamp_remove,cnn_gru_dual_attn_recon_supcon3_dann,DDM_01_06,9544,0.376153,0.644908,0.268755,19.114779


In [19]:
summary = (results_df
    .groupby(["group", "model", "outlier_filter", "curve_alignment", "pc_ttp_anchor", "train_center_frac"])
    .agg(n_chips=("held_out_chip", "nunique"),
        acc_baseline=("acc_baseline", "mean"),
        acc_pc_recenter=("acc_pc_recenter", "mean"),
        recenter_delta=("recenter_delta", "mean"))
    .reset_index()
    .sort_values("acc_baseline", ascending=False))
summary

,group,model,outlier_filter,curve_alignment,pc_ttp_anchor,train_center_frac,n_chips,acc_baseline,acc_pc_recenter,recenter_delta
24,final_4_chip_cov_hadv_iav,cnn_gru_dual_attn_recon_supcon3_coral,noamp_remove,pc_ttp,min,0.5,1,0.547150,0.647946,0.100796
23,final_4_chip_cov_hadv_iav,cnn_gru_dual_attn_recon_supcon3,noamp_remove,pc_ttp,min,0.5,1,0.493451,0.634587,0.141136
21,final_4_chip_cov_hadv_iav,cnn_gru_dual_attn_recon_coral,noamp_remove,pc_ttp,min,0.5,1,0.457146,0.727735,0.270589
22,final_4_chip_cov_hadv_iav,cnn_gru_dual_attn_recon_dann,noamp_remove,pc_ttp,min,0.5,1,0.432995,0.602473,0.169478
3,final_4_chip_clean_nn,cnn_gru_dual_attn_recon_supcon3,noamp_remove,pc_ttp,min,0.5,1,0.418075,0.336262,-0.081813
20,final_4_chip_cov_hadv_iav,cnn_gru_dual_attn_recon,noamp_remove,pc_ttp,min,0.5,1,0.413087,0.605302,0.192215
17,final_4_chip_cleanv2_nn,cnn_gru_dual_attn_recon_supcon3,noamp_remove,pc_ttp,min,0.5,1,0.403605,0.516826,0.113222
15,final_4_chip_cleanv2_nn,cnn_gru_dual_attn_recon_coral,noamp_remove,pc_ttp,min,0.5,1,0.393060,0.198848,-0.194212
25,final_4_chip_cov_hadv_iav,cnn_gru_dual_attn_recon_supcon3_dann,noamp_remove,pc_ttp,min,0.5,1,0.380815,0.625105,0.244290
18,final_4_chip_cleanv2_nn,cnn_gru_dual_attn_recon_supcon3_coral,noamp_remove,pc_ttp,min,0.5,1,0.370588,0.303881,-0.066707


In [20]:
summary.sort_values("recenter_delta", ascending=False)

,group,model,outlier_filter,curve_alignment,pc_ttp_anchor,train_center_frac,n_chips,acc_baseline,acc_pc_recenter,recenter_delta
21,final_4_chip_cov_hadv_iav,cnn_gru_dual_attn_recon_coral,noamp_remove,pc_ttp,min,0.5,1,0.457146,0.727735,0.270589
16,final_4_chip_cleanv2_nn,cnn_gru_dual_attn_recon_dann,noamp_remove,pc_ttp,min,0.5,1,0.163401,0.433780,0.270378
2,final_4_chip_clean_nn,cnn_gru_dual_attn_recon_dann,noamp_remove,pc_ttp,min,0.5,1,0.203298,0.467806,0.264508
25,final_4_chip_cov_hadv_iav,cnn_gru_dual_attn_recon_supcon3_dann,noamp_remove,pc_ttp,min,0.5,1,0.380815,0.625105,0.244290
20,final_4_chip_cov_hadv_iav,cnn_gru_dual_attn_recon,noamp_remove,pc_ttp,min,0.5,1,0.413087,0.605302,0.192215
22,final_4_chip_cov_hadv_iav,cnn_gru_dual_attn_recon_dann,noamp_remove,pc_ttp,min,0.5,1,0.432995,0.602473,0.169478
23,final_4_chip_cov_hadv_iav,cnn_gru_dual_attn_recon_supcon3,noamp_remove,pc_ttp,min,0.5,1,0.493451,0.634587,0.141136
0,final_4_chip_clean_nn,cnn_gru_dual_attn_recon,noamp_remove,pc_ttp,min,0.5,1,0.219339,0.351144,0.131805
17,final_4_chip_cleanv2_nn,cnn_gru_dual_attn_recon_supcon3,noamp_remove,pc_ttp,min,0.5,1,0.403605,0.516826,0.113222
14,final_4_chip_cleanv2_nn,cnn_gru_dual_attn_recon,noamp_remove,pc_ttp,min,0.5,1,0.344788,0.452625,0.107837
